# Sorting review
Reads existing raw Sorting / SortingAnalyzer. No run_sorter call. Review accepted, rejected and borderline candidates; acceptance does not establish a biological neuron. Zero-unit runs have no analyzer: inspect their empty CSVs and stop after the first cell.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from utils import load_config, read_json
config = load_config(ROOT / 'config.yaml')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spikeinterface as si
sorting_folder, qc_folder = Path(config['paths']['sorting']), Path(config['paths']['qc'])
sorting_raw = si.load(sorting_folder / 'sorting_raw')
decisions = pd.read_csv(qc_folder / 'metrics/curation_decisions.csv', index_col='unit_id')
display(decisions)
print('Raw unit IDs:', sorting_raw.unit_ids)
analyzer = si.load(qc_folder / 'analyzer') if (qc_folder / 'analyzer').exists() else None


In [ ]:
# Edit UNIT_INDEX to inspect any candidate in the complete decision table.
if analyzer is None:
    raise ValueError('Zero raw units: no SortingAnalyzer to review.')
UNIT_INDEX = 0
unit = sorting_raw.unit_ids[UNIT_INDEX]
template = analyzer.get_extension('templates').get_unit_template(unit, operator='average')
main = analyzer.get_main_channels(outputs='index')[UNIT_INDEX]
waves = analyzer.get_extension('waveforms').get_waveforms_one_unit(unit)
sparse_channels = analyzer.sparsity.unit_id_to_channel_indices[unit]
local_main = np.flatnonzero(sparse_channels == main)
fs = sorting_raw.get_sampling_frequency()
before = round(analyzer.get_extension('waveforms').params['ms_before']*fs/1000)
time_ms = (np.arange(len(template))-before)/fs*1000
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(time_ms, template[:, main])
if len(waves) and len(local_main):
    axes[1].plot(time_ms, waves[:30, :, local_main[0]].T, alpha=.2)
geometry = analyzer.recording.get_channel_locations()
axes[2].scatter(geometry[:, 0], geometry[:, 1], c=np.ptp(template, axis=0))
axes[0].set(title='Mean template', xlabel='ms', ylabel='uV')
axes[1].set(title='Sampled waveforms', xlabel='ms', ylabel='uV')
axes[2].set(title='Spatial footprint: template PTP', xlabel='x (um)', ylabel='y (um)')
axes[2].set_aspect('equal'); plt.tight_layout()


In [ ]:
times = sorting_raw.get_unit_spike_train(unit)/fs
amplitudes = analyzer.get_extension('spike_amplitudes').get_data(copy=False)
indices = sorting_raw.get_spike_vector_to_indices()
amps = amplitudes[indices[0][unit]]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(times, amps, s=2)
axes[0].set(xlabel='s', ylabel='uV', title='Amplitude over time')
duration = analyzer.recording.get_total_duration()
edges = np.linspace(0, duration, max(2, int(np.ceil(duration))+1))
counts, _ = np.histogram(times, edges)
axes[1].stairs(counts/np.diff(edges), edges)
axes[1].set(xlabel='s', ylabel='Hz', title='Firing rate')
isi_ms = np.diff(times)*1000
axes[2].hist(isi_ms[isi_ms <= 100], bins=np.linspace(0, 100, 101))
axes[2].set(xlabel='ISI (ms)', title='ISI, entire single recording')
plt.tight_layout()


In [ ]:
# Bounded diagnostic ACG: no self pairs; mirror positive lags; label truncation.
half_ms, pair_cap = 50., 200000
lags, pairs, truncated = [], 0, False
for i, stamp in enumerate(times):
    right = np.searchsorted(times, stamp+half_ms/1000, side='right')
    if pairs+right-i-1 > pair_cap:
        truncated = True
        break
    lags.extend((times[i+1:right]-stamp)*1000)
    pairs += right-i-1
lags = np.asarray(lags)
plt.hist(np.concatenate([-lags, lags]), bins=np.arange(-half_ms, half_ms+1))
plt.xlabel('Lag (ms)'); plt.ylabel('Pairs')
plt.title('ACG (truncated)' if truncated else 'ACG')


In [ ]:
# Compare candidates; thresholds operate on SI formulas, not confirmed neuron labels.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for accepted, label in [(True, 'candidate accepted'), (False, 'candidate rejected')]:
    group = decisions[decisions.accepted == accepted]
    for ax, metric in zip(axes, ['snr', 'firing_rate', 'isi_violations_ratio']):
        values = group[metric].to_numpy(dtype=float)
        ax.hist(values[np.isfinite(values)], alpha=.5, label=label)
        ax.set_xlabel(metric)
for ax in axes:
    ax.legend()
plt.tight_layout()
display(decisions[~decisions.accepted][['rejection_reason']])
